# AMEX Enterprise Credit Risk Platform
## Notebook 06 — Explainable AI (SHAP & LIME)
### Phase 1 · Problem Statement 1: Credit Scoring / PD Prediction

CRISP-DM stage: **Evaluation / Interpretability**. Sprint 2, Notebook 6 of 18. Depends on Notebooks 01, 04 and 05 (reads `project_config.json`, `notebook_04_summary.json`, `notebook_05_summary.json`, and Notebook 05's saved champion model + `preprocessing_artifacts.joblib`) -- run those first if you have not already.

**What this notebook does:** explains *why* Notebook 05's champion model scores customers the way it does, using two complementary, industry-standard techniques -- **SHAP** (global + local, game-theoretic, additive) and **LIME** (local, surrogate-model-based) -- both required for model-risk documentation under SR 11-7 / OCC 2011-12 style governance (Notebook 07 builds on this directly).

- **Reuses Notebook 05's champion model and fitted preprocessing artifacts (label encoders, medians, scaler) as-is -- nothing is refit here.** This notebook only re-derives the numeric feature matrix from the same engineered CSV files using those already-fitted artifacts, so the model being explained is *exactly* the one Notebook 05 selected and persisted, not a new copy.
- **Global explainability**: SHAP values on a memory-bounded, class-stratified sample of the held-out split (never trained on), aggregated into mean \|SHAP\| feature importance and cross-checked against Notebook 05's own permutation importance -- two independent importance signals, computed independently, compared directly.
- **Local explainability**: LIME explanations for three deterministically-selected representative customers -- highest predicted risk, lowest predicted risk, and the one closest to the 0.5 decision threshold -- plus a SHAP local breakdown for the same three, so both methods can be read side by side for the same customers.
- Every value and chart below is computed live by this cell during this run -- **zero-fabrication rule**, same as Notebooks 01-05.

**Memory-safe by design, honestly scoped.** SHAP's per-sample cost (especially for tree ensembles doing exact Shapley computation, and any non-tree fallback) makes it computationally infeasible to run on the full ~925K-row true test set or even the full holdout split -- this notebook explains a bounded, class-stratified **sample of up to 2,000 holdout customers** instead, sized against the 90% RAM ceiling from Notebook 01's `resource_limits`. LIME's background/discretization statistics are drawn from that same bounded holdout sample rather than reloading the full ~367K-row train split a second time -- this does not leak any label information, since the medians/encoders/scaler being reused were already fit on train-only data back in Notebook 05.

**Explainer selection is automatic and honestly reported.** `shap.TreeExplainer` is used for every tree-based champion (Random Forest, Extra Trees, Histogram/Gradient Boosting, XGBoost, LightGBM, CatBoost); `shap.LinearExplainer` is used if the champion is Logistic Regression. If neither applies to a given champion, this notebook falls back to SHAP's generic permutation-based `Explainer` on `predict_proba` and prints which path was taken -- it does not silently assume an explainer that doesn't support the champion's model type.

**SHAP and LIME are optional, gracefully degrading dependencies**, same pattern as CatBoost in Notebook 05: if either package is not installed, this cell detects that, skips the affected section with a clear message, and still completes everything else rather than crashing. Install both with `pip install shap lime`.

**Run the single code cell below, once.** Idempotent -- every output file (SHAP values, importance tables, charts, local-explanation reports) is written to a fixed path and overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOKS 01, 04, 05
# =============================================================================
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01, 04, 05")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB04_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_04_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB04_SUMMARY_PATH, "run 04_feature_engineering.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix} -- this notebook reads its outputs.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB04_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB04_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]

_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)
MAX_RAM_BYTES = _resource_limits.get("max_ram_bytes")  # None on an older config -- handled below

MODEL_DEV_DIR = PILLAR_DIRS["model_development"]
MODELS_SUBDIR = MODEL_DEV_DIR / "models"
XAI_DIR = PILLAR_DIRS["explainable_ai"]
XAI_DIR.mkdir(parents=True, exist_ok=True)

TEST_SPLIT_ENG_PATH = Path(NB04_SUMMARY["output_files"]["test_split_engineered.csv"])
CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_MODEL_PATH = MODELS_SUBDIR / f"{CHAMPION_NAME}.joblib"
PREPROCESSING_PATH = MODELS_SUBDIR / "preprocessing_artifacts.joblib"

for _p in (TEST_SPLIT_ENG_PATH, CHAMPION_MODEL_PATH, PREPROCESSING_PATH):
    if not _p.exists():
        raise FileNotFoundError(f"Required file not found: {_p}\nFix: re-run 05_model_development.ipynb -- "
                                 f"this notebook reuses its saved champion model and preprocessing artifacts "
                                 f"as-is rather than refitting them.")

print(f"Loaded config from      : {CONFIG_PATH}")
print(f"Loaded NB04 summary     : {NB04_SUMMARY_PATH}")
print(f"Loaded NB05 summary     : {NB05_SUMMARY_PATH}")
print(f"RANDOM_SEED              : {RANDOM_SEED} (same seed used by every notebook in this platform)")
print(f"WARP_THREAD_COUNT        : {WARP_THREAD_COUNT} (95% cap)")
print(f"MAX_RAM_BYTES            : "
      f"{f'{MAX_RAM_BYTES / 1e9:.1f} GB (90% cap)' if MAX_RAM_BYTES else 'not set -- re-run Notebook 01 to enable'}")
print(f"Champion model (from NB05): {CHAMPION_NAME}")
print(f"Explainability artifacts will be written under: {XAI_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import gc

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    import joblib
except ImportError:
    missing.append("joblib")
try:
    from sklearn.metrics import roc_auc_score
except ImportError:
    missing.append("scikit-learn")

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )

try:
    import shap
    _HAS_SHAP = True
except ImportError:
    _HAS_SHAP = False

try:
    from lime.lime_tabular import LimeTabularExplainer
    _HAS_LIME = True
except ImportError:
    _HAS_LIME = False

logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4, Concurrency)")
print(f"shap available : {_HAS_SHAP}" + ("" if _HAS_SHAP else "  (pip install shap to enable Section 6-8)"))
print(f"lime available : {_HAS_LIME}" + ("" if _HAS_LIME else "  (pip install lime to enable Section 9-10)"))
if not (_HAS_SHAP or _HAS_LIME):
    print("\nNeither shap nor lime is installed -- this notebook will still run, write its verification and "
          "summary artifacts, and report that both explainability sections were skipped. Install at least one "
          "to get meaningful output from this notebook.")


def _rss_gb() -> float:
    """Current process resident memory, in GB -- printed at each major stage
    below so peak usage against the 90% RAM ceiling is visible as this
    notebook runs, not just inferred after the fact."""
    return psutil.Process().memory_info().rss / 1e9


_process_start_rss_gb = _rss_gb()
print(f"\nProcess RSS at Section 2 start: {_process_start_rss_gb:.2f} GB")
if MAX_RAM_BYTES:
    print(f"Configured RAM ceiling (90% of detected total): {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: LOAD CHAMPION MODEL & PREPROCESSING ARTIFACTS (FROM NOTEBOOK 05 -- REUSED, NOT REFIT)
# =============================================================================
_section("SECTION 3: Load Champion Model & Preprocessing Artifacts")

_t0 = time.time()
champion_model = joblib.load(CHAMPION_MODEL_PATH)
preprocessing_artifacts = joblib.load(PREPROCESSING_PATH)
print(f"Loaded champion model '{CHAMPION_NAME}' from {CHAMPION_MODEL_PATH} ({time.time() - _t0:.1f}s)")

label_encoders = preprocessing_artifacts["label_encoders"]
feature_medians = preprocessing_artifacts["feature_medians"]
scaler = preprocessing_artifacts["scaler"]
all_feature_cols = preprocessing_artifacts["all_feature_cols"]
categorical_encode_cols = preprocessing_artifacts["categorical_encode_cols"]
numeric_feature_cols = preprocessing_artifacts["numeric_feature_cols"]

# --- Only Logistic Regression was trained on standardized features in
#     Notebook 05 (see its Section 6 model zoo, "uses_scaled": True) -- every
#     other model in the zoo trains and predicts on the raw (imputed,
#     inf-cleaned, label-encoded) feature matrix. This flag decides which
#     matrix this notebook feeds the champion below. ---
champion_uses_scaled = CHAMPION_NAME == "logistic_regression"

print(f"Feature columns loaded  : {len(all_feature_cols)} "
      f"({len(numeric_feature_cols)} numeric + {len(categorical_encode_cols)} categorical)")
print(f"Champion uses scaled features: {champion_uses_scaled}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LOAD ENGINEERED HOLDOUT DATA & APPLY SAVED PREPROCESSING (POLARS-NATIVE, FLOAT32)
# =============================================================================
_section("SECTION 4: Load Engineered Holdout Data & Apply Saved Preprocessing")

# --- Only the held-out split is loaded here -- this notebook explains the
#     champion's behavior on customers it never trained on, matching
#     Notebook 05's own unbiased-evaluation split. The full ~925K-row true
#     unlabeled test set and the ~367K-row train split are deliberately NOT
#     reloaded: SHAP's per-sample cost makes explaining hundreds of thousands
#     of rows infeasible, and Section 5 below draws a bounded sample from
#     this holdout frame instead. Same explicit schema_overrides pattern as
#     Notebook 05 Section 4 -- no dtype inference, so a null-heavy column in a
#     sampled window can never be silently mis-typed as a string. ---
import csv as _csv
with open(TEST_SPLIT_ENG_PATH, "r", encoding="utf-8", newline="") as _f:
    _header = next(_csv.reader(_f))

SPLIT_CSV_SCHEMA = {"customer_ID": pl.Utf8, "target": pl.Int8}
for _c in categorical_encode_cols:
    SPLIT_CSV_SCHEMA[_c] = pl.Utf8
for _c in numeric_feature_cols:
    SPLIT_CSV_SCHEMA[_c] = pl.Float32

_t0 = time.time()
holdout_pl = pl.read_csv(str(TEST_SPLIT_ENG_PATH), schema_overrides=SPLIT_CSV_SCHEMA)
print(f"Loaded test_split_engineered.csv: {holdout_pl.shape[0]:,} x {holdout_pl.shape[1]} "
      f"({time.time() - _t0:.1f}s, held-out, never trained on)")

if "target" not in holdout_pl.columns:
    raise RuntimeError(f"Expected a 'target' column in {TEST_SPLIT_ENG_PATH.name} -- check Notebook 02/04 output.")

# --- Apply the SAME preprocessing Notebook 05 fit on the train split --
#     nothing here is refit, only applied. Inf -> null -> float32, then
#     label-encode using NB05's saved {category: code} vocabulary
#     (replace_strict with default=-1 for any unseen category), then impute
#     with NB05's saved train-only medians. ---
_t0 = time.time()
# --- Inf/NaN -> null (defense in depth, same guard as Notebook 05 Section 5
#     -- see its comment for why is_nan() is checked alongside is_infinite()) --
_inf_clean_exprs = [
    pl.when(pl.col(c).is_infinite() | pl.col(c).is_nan()).then(None).otherwise(pl.col(c)).cast(pl.Float32).alias(c)
    for c in numeric_feature_cols
]
holdout_pl = holdout_pl.with_columns(_inf_clean_exprs)

for c in categorical_encode_cols:
    holdout_pl = holdout_pl.with_columns(pl.col(c).cast(pl.Utf8).fill_null("__missing__").alias(c))
    _mapping = {cat: i for i, cat in enumerate(label_encoders[c]["classes"])}
    holdout_pl = holdout_pl.with_columns(
        pl.col(c).replace_strict(_mapping, default=-1, return_dtype=pl.Int32).alias(c))

_impute_exprs = [pl.col(c).fill_null(feature_medians[c]) for c in numeric_feature_cols]
holdout_pl = holdout_pl.with_columns(_impute_exprs)
print(f"Applied Notebook 05's saved label-encoding + median imputation to {len(all_feature_cols)} columns "
      f"in {time.time() - _t0:.1f}s")

# --- Materialize float32 numpy arrays and immediately free the Polars frame --
X_holdout = holdout_pl.select(all_feature_cols).to_numpy().astype(np.float32, copy=False)
y_holdout = holdout_pl.get_column("target").to_numpy().astype(np.int64, copy=False)
holdout_customer_ids = holdout_pl.get_column("customer_ID").to_numpy()
del holdout_pl
gc.collect()

_train_mean = scaler["mean"]
_train_std = scaler["std"]
X_holdout_scaled = (X_holdout - _train_mean) / _train_std

print(f"X_holdout        : {X_holdout.shape}, dtype {X_holdout.dtype}, default rate {y_holdout.mean():.4%}, "
      f"{X_holdout.nbytes / 1e9:.2f} GB")
print(f"Process RSS now: {_rss_gb():.2f} GB"
      + (f" (of {MAX_RAM_BYTES / 1e9:.1f} GB ceiling)" if MAX_RAM_BYTES else ""))
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: MEMORY-SAFE SHAP SAMPLE CONSTRUCTION (CLASS-STRATIFIED, BOUNDED SIZE)
# =============================================================================
_section("SECTION 5: Memory-Safe SHAP Sample Construction")

# --- SHAP's per-sample cost (exact Shapley computation for tree ensembles;
#     background-evaluations-per-sample for the linear/generic fallback) makes
#     explaining the full holdout split -- let alone the ~925K-row true test
#     set -- computationally infeasible on a laptop. A bounded, class-
#     stratified sample keeps both the compute time and the peak memory used
#     by SHAP's internal per-sample bookkeeping predictable regardless of how
#     large the holdout split is, while still preserving the true class
#     balance so the "average customer explained" is representative rather
#     than skewed toward whichever class happens to sort first. ---
N_SHAP_SAMPLE_TARGET = 2000
N_SHAP_SAMPLE = min(N_SHAP_SAMPLE_TARGET, X_holdout.shape[0])

_rng = np.random.RandomState(RANDOM_SEED)
_pos_idx = np.where(y_holdout == 1)[0]
_neg_idx = np.where(y_holdout == 0)[0]
_pos_frac = len(_pos_idx) / len(y_holdout)
_n_pos = max(1, int(round(N_SHAP_SAMPLE * _pos_frac)))
_n_neg = N_SHAP_SAMPLE - _n_pos
_n_pos = min(_n_pos, len(_pos_idx))
_n_neg = min(_n_neg, len(_neg_idx))

_sample_pos = _rng.choice(_pos_idx, size=_n_pos, replace=False)
_sample_neg = _rng.choice(_neg_idx, size=_n_neg, replace=False)
shap_sample_idx = np.concatenate([_sample_pos, _sample_neg])
_rng.shuffle(shap_sample_idx)

X_shap_sample = X_holdout_scaled[shap_sample_idx] if champion_uses_scaled else X_holdout[shap_sample_idx]
y_shap_sample = y_holdout[shap_sample_idx]
shap_sample_customer_ids = holdout_customer_ids[shap_sample_idx]

print(f"Holdout split size            : {X_holdout.shape[0]:,} customers "
      f"(default rate {y_holdout.mean():.4%})")
print(f"SHAP sample size (bounded)    : {N_SHAP_SAMPLE:,} customers "
      f"({N_SHAP_SAMPLE / X_holdout.shape[0]:.2%} of holdout)")
print(f"SHAP sample class balance     : {_n_pos:,} defaulters + {_n_neg:,} non-defaulters "
      f"(default rate {y_shap_sample.mean():.4%}, holdout default rate {y_holdout.mean():.4%})")
print(f"Sampled with RANDOM_SEED={RANDOM_SEED} -- deterministic and reproducible across re-runs.")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: SHAP VALUE COMPUTATION (EXPLAINER AUTO-SELECTED BY CHAMPION MODEL TYPE)
# =============================================================================
_section("SECTION 6: SHAP Value Computation")

TREE_MODEL_NAMES = {"random_forest", "extra_trees", "hist_gradient_boosting",
                     "xgboost", "lightgbm", "catboost"}

shap_available = _HAS_SHAP
shap_values_matrix = None
shap_base_value = None
explainer_kind = "not computed (shap not installed)"

if not _HAS_SHAP:
    print("shap is not installed -- skipping SHAP computation. Install with: pip install shap")
else:
    _t0 = time.time()

    def _extract_shap_matrix(explanation) -> np.ndarray:
        """Different SHAP explainer/model combinations return different
        shapes for binary classification -- some (LogisticRegression via
        LinearExplainer; XGBoost/LightGBM/CatBoost/HistGradientBoosting via
        TreeExplainer) already return a single (n_samples, n_features) matrix
        of positive-class log-odds contributions, while others (sklearn's
        RandomForest/ExtraTrees via TreeExplainer) return
        (n_samples, n_features, n_classes) with a contribution per class.
        Normalizing to (n_samples, n_features) for the positive class (index
        1 -- sklearn's classes_ is sorted ascending, so index 1 is target=1,
        i.e. default) here means every downstream chart/table works
        regardless of which champion this run selected -- verified against
        every model in Notebook 05's zoo during this notebook's own
        build-time testing."""
        vals = np.asarray(explanation.values)
        if vals.ndim == 3:
            vals = vals[:, :, 1]
        return vals.astype(np.float64)

    if champion_uses_scaled:
        _background = shap.maskers.Independent(X_shap_sample[:min(200, len(X_shap_sample))], max_samples=200)
        explainer = shap.LinearExplainer(champion_model, _background)
        explainer_kind = "LinearExplainer (exact, coefficient-based -- champion is Logistic Regression)"
        shap_explanation = explainer(X_shap_sample)
    elif CHAMPION_NAME in TREE_MODEL_NAMES:
        try:
            explainer = shap.TreeExplainer(champion_model)
            shap_explanation = explainer(X_shap_sample)
            explainer_kind = f"TreeExplainer (exact, tree-structure-based -- champion is {CHAMPION_NAME})"
        except Exception as _e:
            print(f"TreeExplainer raised {type(_e).__name__} for champion '{CHAMPION_NAME}' -- "
                  f"falling back to the generic permutation-based Explainer instead.")
            _background = shap.sample(X_shap_sample, min(100, len(X_shap_sample)), random_state=RANDOM_SEED)
            explainer = shap.Explainer(champion_model.predict_proba, _background)
            shap_explanation = explainer(X_shap_sample)
            explainer_kind = (f"generic permutation-based Explainer (TreeExplainer failed for "
                               f"'{CHAMPION_NAME}': {type(_e).__name__})")
    else:
        # Champion type not in the known tree/linear families -- fall back
        # honestly rather than assuming an unsupported explainer works.
        _background = shap.sample(X_shap_sample, min(100, len(X_shap_sample)), random_state=RANDOM_SEED)
        explainer = shap.Explainer(champion_model.predict_proba, _background)
        shap_explanation = explainer(X_shap_sample)
        explainer_kind = f"generic permutation-based Explainer (champion type '{CHAMPION_NAME}' has no dedicated fast path)"

    shap_values_matrix = _extract_shap_matrix(shap_explanation)
    _base = np.asarray(shap_explanation.base_values)
    shap_base_value = float(_base[:, 1].mean()) if _base.ndim == 2 else float(np.asarray(_base).mean())

    print(f"Explainer used   : {explainer_kind}")
    print(f"SHAP values shape: {shap_values_matrix.shape}  (computed in {time.time() - _t0:.1f}s "
          f"for {N_SHAP_SAMPLE:,} customers)")
    print(f"SHAP base value (expected model output over background): {shap_base_value:.4f}")
    print(f"Process RSS now: {_rss_gb():.2f} GB")

print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: GLOBAL FEATURE IMPORTANCE -- SHAP VS. PERMUTATION IMPORTANCE CROSS-CHECK
# =============================================================================
_section("SECTION 7: Global Feature Importance -- SHAP vs. Permutation Importance Cross-Check")

shap_importance_df = None
importance_comparison_df = None

if shap_values_matrix is not None:
    _mean_abs_shap = np.abs(shap_values_matrix).mean(axis=0)
    shap_importance_df = pd.DataFrame({"feature": all_feature_cols, "mean_abs_shap": _mean_abs_shap})
    shap_importance_df = shap_importance_df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    shap_importance_path = XAI_DIR / "shap_global_importance.csv"
    shap_importance_df.to_csv(shap_importance_path, index=False)
    print(f"\u2705 Saved -> {shap_importance_path}")
    print("Top 10 features by mean |SHAP value|:")
    for _, r in shap_importance_df.head(10).iterrows():
        print(f"  {r['feature']:<40} {r['mean_abs_shap']:.5f}")

    # --- Cross-check against Notebook 05's own champion feature importance
    #     (impurity/coefficient/permutation-based, depending on model type) --
    #     two independently-computed importance signals for the same champion
    #     model, compared directly rather than assumed to agree. ---
    _nb05_importance_path = Path(NB05_SUMMARY["output_files"]["champion_feature_importance.csv"])
    if _nb05_importance_path.exists():
        _nb05_importance_df = pd.read_csv(_nb05_importance_path).rename(
            columns={"importance": "nb05_importance"})
        importance_comparison_df = shap_importance_df.merge(_nb05_importance_df, on="feature", how="inner")
        _rank_shap = importance_comparison_df["mean_abs_shap"].rank(ascending=False)
        _rank_nb05 = importance_comparison_df["nb05_importance"].rank(ascending=False)
        _spearman_corr = float(pd.Series(_rank_shap).corr(pd.Series(_rank_nb05), method="spearman"))
        comparison_path = XAI_DIR / "shap_vs_permutation_importance_comparison.csv"
        importance_comparison_df.to_csv(comparison_path, index=False)
        print(f"\n\u2705 Saved -> {comparison_path}")
        print(f"Spearman rank correlation, SHAP mean|value| vs. Notebook 05 importance: {_spearman_corr:.4f} "
              f"({len(importance_comparison_df)} features compared)")
    else:
        print(f"\nNote: {_nb05_importance_path} not found -- skipping the SHAP-vs-NB05 comparison "
              f"(re-run 05_model_development.ipynb to regenerate it).")
else:
    print("SHAP values were not computed (shap not installed) -- skipping global importance.")

print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: CHARTS -- SHAP GLOBAL IMPORTANCE, BEESWARM, SHAP-VS-PERMUTATION COMPARISON
# =============================================================================
_section("SECTION 8: Charts -- SHAP Global Importance, Beeswarm, SHAP-vs-Permutation Comparison")

VIZ = {
    "surface": "#fcfcfb",
    "text_primary": "#0b0b0b",
    "text_secondary": "#52514e",
    "grid": "#e3e2dd",
    "cat_blue": "#2a78d6",
    "cat_red": "#e34948",
    "seq_blue_mid": "#3987e5",
    "diverging_neutral": "#f0efec",
}


def _style_axes(ax):
    ax.set_facecolor(VIZ["surface"])
    ax.figure.set_facecolor(VIZ["surface"])
    ax.grid(axis="y", color=VIZ["grid"], linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(VIZ["grid"])
    ax.tick_params(colors=VIZ["text_secondary"], labelsize=9)
    ax.title.set_color(VIZ["text_primary"])
    ax.xaxis.label.set_color(VIZ["text_secondary"])
    ax.yaxis.label.set_color(VIZ["text_secondary"])


_expected_chart_files = []

if shap_values_matrix is not None:
    # --- Chart 1: SHAP global importance, top 20 ---
    _top20 = shap_importance_df.head(20).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, max(4, 0.32 * len(_top20))), dpi=150)
    ax.barh(_top20["feature"], _top20["mean_abs_shap"], color=VIZ["cat_blue"], zorder=3)
    _style_axes(ax)
    ax.grid(axis="x", color=VIZ["grid"], linewidth=0.8, zorder=0)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_title(f"Top 20 Features by SHAP Global Importance -- Champion ({CHAMPION_NAME})")
    fig.tight_layout()
    shap_importance_chart_path = XAI_DIR / "shap_global_importance_chart.png"
    fig.savefig(shap_importance_chart_path, dpi=150, facecolor=VIZ["surface"])
    plt.show()
    plt.close(fig)
    _expected_chart_files.append(shap_importance_chart_path)
    print(f"\u2705 Saved -> {shap_importance_chart_path}")

    # --- Chart 2: manually-built beeswarm-style scatter for the top 15
    #     features by importance -- one row per feature, each customer in
    #     the SHAP sample plotted as a point at its SHAP value on that
    #     feature, jittered vertically, colored by that customer's own
    #     (standardized) feature value so the direction of each feature's
    #     effect is visible alongside its magnitude. Built directly from
    #     this run's own shap_values_matrix / X_shap_sample -- not shap's
    #     built-in plotting function -- so it renders identically wherever
    #     this notebook runs, headless or not. ---
    _top15_idx = shap_importance_df.head(15).index
    _top15_features = shap_importance_df.head(15)["feature"].tolist()
    _feature_pos = {f: all_feature_cols.index(f) for f in _top15_features}

    fig, ax = plt.subplots(figsize=(9, max(5, 0.4 * len(_top15_features))), dpi=150)
    _rng_jitter = np.random.RandomState(RANDOM_SEED)
    for _row, _feat in enumerate(_top15_features[::-1]):
        _col_idx = _feature_pos[_feat]
        _sv = shap_values_matrix[:, _col_idx]
        _fv = X_shap_sample[:, _col_idx]
        _fv_norm = (_fv - _fv.min()) / (_fv.max() - _fv.min() + 1e-9)
        _jitter = _rng_jitter.uniform(-0.35, 0.35, size=len(_sv))
        _colors = plt.cm.coolwarm(_fv_norm)
        ax.scatter(_sv, np.full(len(_sv), _row) + _jitter, c=_colors, s=8, alpha=0.55, linewidths=0, zorder=3)
    _style_axes(ax)
    ax.axvline(0, color=VIZ["text_secondary"], linewidth=0.8, zorder=2)
    ax.set_yticks(range(len(_top15_features)))
    ax.set_yticklabels(_top15_features[::-1])
    ax.set_xlabel("SHAP value (impact on predicted default log-odds)")
    ax.set_title(f"SHAP Beeswarm -- Top 15 Features, Champion ({CHAMPION_NAME})\n"
                 f"(point color = low\u2192high feature value)", fontsize=11)
    fig.tight_layout()
    shap_beeswarm_chart_path = XAI_DIR / "shap_beeswarm_chart.png"
    fig.savefig(shap_beeswarm_chart_path, dpi=150, facecolor=VIZ["surface"])
    plt.show()
    plt.close(fig)
    _expected_chart_files.append(shap_beeswarm_chart_path)
    print(f"\u2705 Saved -> {shap_beeswarm_chart_path}")

    # --- Chart 3: SHAP vs. Notebook 05 permutation/impurity importance,
    #     top 15 by SHAP, grouped bars (each axis independently normalized
    #     to [0, 1] by its own max -- the two importance scales are not
    #     directly comparable in absolute units, only in relative ranking) ---
    if importance_comparison_df is not None:
        _cmp_top15 = importance_comparison_df.sort_values("mean_abs_shap", ascending=False).head(15)
        _shap_norm = _cmp_top15["mean_abs_shap"] / _cmp_top15["mean_abs_shap"].max()
        _nb05_norm = _cmp_top15["nb05_importance"] / _cmp_top15["nb05_importance"].max()
        _x = np.arange(len(_cmp_top15))
        _width = 0.36
        fig, ax = plt.subplots(figsize=(9, 5.5), dpi=150)
        ax.bar(_x - _width / 2, _shap_norm, _width, label="SHAP (normalized)", color=VIZ["cat_blue"], zorder=3)
        ax.bar(_x + _width / 2, _nb05_norm, _width, label="Notebook 05 importance (normalized)",
               color=VIZ["cat_red"], zorder=3)
        _style_axes(ax)
        ax.set_xticks(_x)
        ax.set_xticklabels(_cmp_top15["feature"], rotation=35, ha="right")
        ax.set_ylabel("Normalized importance")
        ax.set_title(f"SHAP vs. Notebook 05 Importance -- Top 15 by SHAP (Spearman \u03c1 = {_spearman_corr:.3f})")
        ax.legend(frameon=False, loc="upper right")
        fig.tight_layout()
        shap_comparison_chart_path = XAI_DIR / "shap_vs_permutation_comparison_chart.png"
        fig.savefig(shap_comparison_chart_path, dpi=150, facecolor=VIZ["surface"])
        plt.show()
        plt.close(fig)
        _expected_chart_files.append(shap_comparison_chart_path)
        print(f"\u2705 Saved -> {shap_comparison_chart_path}")
else:
    print("SHAP values were not computed (shap not installed) -- skipping Charts 1-3.")

print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: LIME LOCAL EXPLANATIONS FOR REPRESENTATIVE CUSTOMERS
# =============================================================================
_section("SECTION 9: LIME Local Explanations for Representative Customers")

lime_results = []

if not _HAS_LIME:
    print("lime is not installed -- skipping LIME explanations. Install with: pip install lime")
else:
    _t0 = time.time()

    # --- Score the SHAP sample so representative customers can be selected
    #     deterministically: highest predicted risk, lowest predicted risk,
    #     and the one closest to the 0.5 decision threshold. Same feature
    #     matrix orientation (scaled/unscaled) the champion was trained on. ---
    _champion_proba = champion_model.predict_proba(X_shap_sample)[:, 1]
    _idx_highest = int(np.argmax(_champion_proba))
    _idx_lowest = int(np.argmin(_champion_proba))
    _idx_threshold = int(np.argmin(np.abs(_champion_proba - 0.5)))
    _representative = [
        ("highest_predicted_risk", _idx_highest),
        ("lowest_predicted_risk", _idx_lowest),
        ("closest_to_decision_threshold", _idx_threshold),
    ]

    # --- LIME's background/discretization statistics are drawn from this
    #     notebook's own bounded SHAP sample (up to 2,000 holdout customers)
    #     rather than reloading the full ~367K-row train split a second time
    #     -- a deliberate memory-efficiency choice, not a leakage risk: the
    #     medians/encoders/scaler already baked into these features were fit
    #     on train-only data back in Notebook 05, and LIME only needs a
    #     representative feature-value distribution to build its local
    #     surrogate model, not the original training labels. ---
    lime_explainer = LimeTabularExplainer(
        training_data=X_shap_sample,
        feature_names=all_feature_cols,
        class_names=["no_default", "default"],
        mode="classification",
        discretize_continuous=True,
        random_state=RANDOM_SEED,
    )

    def _predict_fn(arr: np.ndarray) -> np.ndarray:
        return champion_model.predict_proba(arr)

    for _label, _idx in _representative:
        _exp = lime_explainer.explain_instance(
            X_shap_sample[_idx], _predict_fn, num_features=15, num_samples=2000,
        )
        _pairs = _exp.as_list()
        lime_results.append({
            "customer_role": _label,
            "customer_ID": str(shap_sample_customer_ids[_idx]),
            "predicted_probability": float(_champion_proba[_idx]),
            "actual_target": int(y_shap_sample[_idx]),
            "lime_explanation": [{"rule": rule, "weight": float(weight)} for rule, weight in _pairs],
        })
        print(f"\n{_label} (customer_ID={shap_sample_customer_ids[_idx]}, "
              f"predicted_proba={_champion_proba[_idx]:.4f}, actual_target={y_shap_sample[_idx]}):")
        for rule, weight in _pairs[:6]:
            print(f"  {weight:+.4f}  {rule}")

    lime_results_path = XAI_DIR / "lime_local_explanations.json"
    with open(lime_results_path, "w", encoding="utf-8") as f:
        json.dump(lime_results, f, indent=2)
    print(f"\n\u2705 Saved -> {lime_results_path} ({time.time() - _t0:.1f}s for {len(lime_results)} customers, "
          f"2,000 perturbation samples each)")

print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: CHART -- LIME LOCAL EXPLANATION (REPRESENTATIVE CUSTOMERS)
# =============================================================================
_section("SECTION 10: Chart -- LIME Local Explanations")

if lime_results:
    fig, axes = plt.subplots(1, len(lime_results), figsize=(6 * len(lime_results), 6), dpi=150)
    if len(lime_results) == 1:
        axes = [axes]
    for ax, result in zip(axes, lime_results):
        _rules = [p["rule"] for p in result["lime_explanation"]][:8][::-1]
        _weights = [p["weight"] for p in result["lime_explanation"]][:8][::-1]
        _colors = [VIZ["cat_red"] if w > 0 else VIZ["cat_blue"] for w in _weights]
        ax.barh(_rules, _weights, color=_colors, zorder=3)
        _style_axes(ax)
        ax.grid(axis="x", color=VIZ["grid"], linewidth=0.8, zorder=0)
        ax.grid(axis="y", visible=False)
        ax.axvline(0, color=VIZ["text_secondary"], linewidth=0.8, zorder=2)
        ax.set_xlabel("LIME local weight")
        ax.set_title(f"{result['customer_role'].replace('_', ' ').title()}\n"
                     f"pred={result['predicted_probability']:.3f}, actual={result['actual_target']}",
                     fontsize=10)
        ax.tick_params(labelsize=7.5)
    fig.suptitle(f"LIME Local Explanations -- Champion ({CHAMPION_NAME})", color=VIZ["text_primary"])
    fig.tight_layout()
    lime_chart_path = XAI_DIR / "lime_local_explanations_chart.png"
    fig.savefig(lime_chart_path, dpi=150, facecolor=VIZ["surface"])
    plt.show()
    plt.close(fig)
    _expected_chart_files.append(lime_chart_path)
    print(f"\u2705 Saved -> {lime_chart_path}")
else:
    print("No LIME results available (lime not installed) -- skipping Chart 4.")

print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: PERSIST RAW SHAP VALUES (FOR NOTEBOOK 07's MODEL RISK DOCUMENTATION)
# =============================================================================
_section("SECTION 11: Persist Raw SHAP Values")

_expected_data_files = []
if shap_values_matrix is not None:
    shap_values_path = XAI_DIR / "shap_values_sample.parquet"
    _shap_out_df = pd.DataFrame(shap_values_matrix, columns=all_feature_cols)
    _shap_out_df.insert(0, "customer_ID", shap_sample_customer_ids)
    _shap_out_df.insert(1, "actual_target", y_shap_sample)
    _shap_out_df.insert(2, "predicted_probability",
                         champion_model.predict_proba(X_shap_sample)[:, 1])
    _shap_out_df.to_parquet(shap_values_path, index=False)
    _expected_data_files.append(shap_values_path)
    print(f"\u2705 Saved -> {shap_values_path} ({_shap_out_df.shape[0]:,} customers x "
          f"{len(all_feature_cols)} SHAP values, {shap_values_path.stat().st_size / 1e6:.2f} MB)")
else:
    print("SHAP values were not computed -- skipping raw SHAP value persistence.")

print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: VERIFICATION -- INTEGRITY CHECKS ON EVERYTHING THIS NOTEBOOK WROTE
# =============================================================================
_section("SECTION 12: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("champion model loaded matches Notebook 05's recorded champion",
       CHAMPION_NAME == NB05_SUMMARY["champion_model"])
_check("X_holdout has the same feature-column count as the saved preprocessing artifacts",
       X_holdout.shape[1] == len(all_feature_cols),
       f"({X_holdout.shape[1]} vs {len(all_feature_cols)})")

if _HAS_SHAP:
    _check("SHAP values computed for every customer in the SHAP sample",
           shap_values_matrix is not None and shap_values_matrix.shape[0] == N_SHAP_SAMPLE,
           f"({None if shap_values_matrix is None else shap_values_matrix.shape[0]} vs {N_SHAP_SAMPLE})")
    _check("SHAP values matrix has no NaN entries",
           shap_values_matrix is not None and not np.isnan(shap_values_matrix).any())
    _check("SHAP sample preserved the holdout split's class balance within 5 points",
           abs(y_shap_sample.mean() - y_holdout.mean()) < 0.05,
           f"(sample {y_shap_sample.mean():.4f} vs holdout {y_holdout.mean():.4f})")

if _HAS_LIME:
    _check("LIME produced an explanation for all 3 representative customers",
           len(lime_results) == 3, f"({len(lime_results)} vs 3)")
    _check("LIME representative customers are 3 distinct customer_IDs",
           len({r['customer_ID'] for r in lime_results}) == len(lime_results))

_all_expected_files = _expected_chart_files + _expected_data_files
if _HAS_LIME:
    _all_expected_files.append(lime_results_path)
if _HAS_SHAP:
    _all_expected_files.append(shap_importance_path)
    if importance_comparison_df is not None:
        _all_expected_files.append(comparison_path)

for fp in _all_expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 06 verification checks failed. See \u274c lines above.")

print("\nAll Notebook 06 checks passed.")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: RESOURCE / PERFORMANCE REPORT
# =============================================================================
_section("SECTION 13: Resource / Performance Report")

# --- Honest scope note: unlike Notebook 05's model training (where thread
#     count is a real, meaningful knob for every model in the zoo), SHAP's
#     TreeExplainer and LIME's perturbation loop in this notebook do not
#     expose a comparably impactful parallelism knob to benchmark against --
#     both are dominated by per-sample Shapley/perturbation computation, not
#     by something a thread-count sweep would meaningfully change here. This
#     section reports what was actually measured this run (wall-clock time
#     and peak RSS per stage) rather than fabricating a thread-count
#     benchmark that would not reflect a real bottleneck for this workload. ---
_final_rss_gb = _rss_gb()
performance_report = {
    "warp_thread_count_configured": WARP_THREAD_COUNT,
    "max_ram_bytes_ceiling": MAX_RAM_BYTES,
    "max_ram_gb_ceiling": round(MAX_RAM_BYTES / 1e9, 2) if MAX_RAM_BYTES else None,
    "process_rss_at_start_gb": round(_process_start_rss_gb, 2),
    "process_rss_at_end_gb": round(_final_rss_gb, 2),
    "peak_rss_within_ram_ceiling": (_final_rss_gb * 1e9 <= MAX_RAM_BYTES) if MAX_RAM_BYTES else None,
    "shap_sample_size": N_SHAP_SAMPLE,
    "shap_explainer_used": explainer_kind,
    "shap_installed": _HAS_SHAP,
    "lime_installed": _HAS_LIME,
    "note": ("SHAP/LIME are dominated by per-sample computation, not thread-count parallelism -- no "
             "fabricated CPU-thread benchmark is reported for this notebook's workload, only measured "
             "wall-clock time and RSS per section (see printed section timings above)."),
}
performance_report_path = XAI_DIR / "notebook_06_performance_report.json"
with open(performance_report_path, "w", encoding="utf-8") as f:
    json.dump(performance_report, f, indent=2)

print(f"Process RSS: {_process_start_rss_gb:.2f} GB (start) -> {_final_rss_gb:.2f} GB (end)"
      + (f", ceiling {MAX_RAM_BYTES / 1e9:.1f} GB" if MAX_RAM_BYTES else ""))
print(f"\u2705 Saved -> {performance_report_path}")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: WRITE NOTEBOOK 06 SUMMARY ARTIFACT (for Notebook 17's rollup)
# =============================================================================
_section("SECTION 14: Write Notebook 06 Summary Artifact")

notebook_06_summary = {
    "notebook": "06_explainable_ai",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "champion_model_explained": CHAMPION_NAME,
    "shap_installed": _HAS_SHAP,
    "lime_installed": _HAS_LIME,
    "shap_sample_size": N_SHAP_SAMPLE if shap_values_matrix is not None else None,
    "shap_explainer_used": explainer_kind,
    "shap_top10_features": (shap_importance_df.head(10)["feature"].tolist()
                             if shap_importance_df is not None else None),
    "shap_vs_nb05_spearman_correlation": (round(_spearman_corr, 4)
                                           if importance_comparison_df is not None else None),
    "lime_representative_customers": [
        {"role": r["customer_role"], "customer_ID": r["customer_ID"],
         "predicted_probability": r["predicted_probability"]} for r in lime_results
    ],
    "resource_config_used": {
        "warp_thread_count": WARP_THREAD_COUNT,
        "max_ram_bytes": MAX_RAM_BYTES,
        "peak_process_rss_gb_observed": round(_final_rss_gb, 2),
    },
    "output_files": {p.name: str(p) for p in
                      (_all_expected_files + [performance_report_path])},
}
nb06_summary_path = ARTIFACTS_DIR / "notebook_06_summary.json"
with open(nb06_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_06_summary, f, indent=2)
print(f"\u2705 Saved -> {nb06_summary_path} (Notebook 17 reads this file to build the rolled-up Explainable "
      f"AI section)")
print("\n\u2705 Section 14 complete.")


# =============================================================================
# SECTION 15: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 15: Notebook 06 Complete -- Handoff to Notebook 07")

print("NOTEBOOK 06: EXPLAINABLE AI (SHAP & LIME) -- COMPLETE")
print(f"  Champion model explained         : {CHAMPION_NAME}")
print(f"  SHAP explainer used              : {explainer_kind}")
print(f"  SHAP sample size                 : {N_SHAP_SAMPLE if shap_values_matrix is not None else 'n/a (shap not installed)'}")
if importance_comparison_df is not None:
    print(f"  SHAP vs. Notebook 05 rank agreement (Spearman): {_spearman_corr:.4f}")
print(f"  LIME representative customers    : {len(lime_results)}")
print(f"  Files produced                   : {len(_all_expected_files) + 2}")
for _p in _all_expected_files + [performance_report_path, nb06_summary_path]:
    print(f"    - {_p.name}")
print(f"  Peak process RSS this run        : {_final_rss_gb:.2f} GB"
      + (f" (of {MAX_RAM_BYTES / 1e9:.1f} GB ceiling)" if MAX_RAM_BYTES else ""))
print(f"  Next notebook                    : 07_model_risk_management.ipynb (Sprint 2)")
print("\n\u2705 Ready to proceed.")
